Based on the video, here is the breakdown of **Edge Nodes** and **Deployment Modes**.

### **1. What is an Edge Node?**
In a professional setting, you (the developer) do not communicate directly with the **Resource Manager** (the "Boss") for security and access reasons.
*   Instead, you log into a "Gateway" machine called an **Edge Node**.
*   You submit your code to the Edge Node, and the Edge Node communicates with the Cluster Manager to request resources.

### **2. Deployment Modes: Client vs. Cluster**
Once you submit your code from the Edge Node, you have two ways to run the **Driver** (the "Team Lead"):

*   **Client Mode (For Development):**
    *   The Driver runs on **your machine** (the Edge Node).
    *   **Pros:** You can see logs and outputs immediately on your screen. Great for debugging.
    *   **Cons:** If you shut down your laptop or lose internet connection, the **Driver dies**, and the entire application fails.
*   **Cluster Mode (For Production):**
    *   The Driver runs on a **Worker Node** inside the cluster.
    *   **Pros:** It is independent of your machine. You can submit the job and close your laptop; the job will keep running.
    *   **Cons:** Harder to debug since logs are stored on the cluster, not your screen.

**Does this distinction make sense?** If so, we can move on to **Partition Pruning** to see how Spark optimizes reading data.

# Types of Clusters Available in Spark

## 1. Standalone Cluster
Spark's built-in cluster manager.

## 2. YARN Cluster
Hadoop's cluster manager.

## 3. Mesos Cluster
Apache Mesos cluster manager.

## 4. Kubernetes Cluster
Container orchestration platform.

---

# Relationship with Client Mode and Cluster Mode

## Client Mode
- The Spark driver runs on the machine where the application is submitted (e.g., Edge Node).
- The cluster manager (Standalone, YARN, Mesos, Kubernetes) allocates resources for executors, but the driver stays local.

## Cluster Mode
- The Spark driver runs inside the cluster (on a worker node).
- The cluster manager allocates resources for both the driver and executors.
- The application is independent of the submitting machine.

---

# Detailed Explanation

- In Client Mode, logs and outputs are visible on the submitting machine, making it suitable for development and debugging.
- If the submitting machine disconnects, the application stops.

- In Cluster Mode, the application continues running even if the submitting machine disconnects.
- Logs are stored on the cluster.
- Both modes are supported by all cluster managers (Standalone, YARN, Mesos, Kubernetes).

---

# Job Clusters vs. All-Purpose Clusters in Apache Spark

- A job cluster refers to a temporary cluster spun up specifically to run a single job or notebook, then terminated once the job completes.
- An all-purpose cluster stays running and can handle multiple jobs, notebooks, or interactive sessions.

- Job clusters are commonly used in managed environments like Databricks, where you submit a job (notebook or script) and Databricks automatically creates a dedicated cluster for that job.
- This ensures resource isolation, reproducibility, and cost efficiency.

---

# Relationship to Client Mode and Cluster Mode

## Client Mode
- The Spark driver runs on the machine where you submit the application (e.g., your local machine or an edge node).
- In Databricks, job clusters typically do not use client mode, because the driver needs to run within the managed environment for reliability.

## Cluster Mode
- The Spark driver runs inside the cluster (on a worker node).
- Job clusters in Databricks always use cluster mode: the driver is launched inside the ephemeral cluster, ensuring the job is independent of the submitting machine and can run reliably even if the user disconnects.

---

# Summary Table

| Cluster Type   | Driver Location         | Mode Used      | Typical Use Case           |
|----------------|------------------------|---------------|----------------------------|
| All-purpose    | Edge node or cluster   | Client/Cluster| Interactive, multi-user    |
| Job cluster    | Cluster worker node    | Cluster       | Single job, automation     |

---

- Job clusters are tightly coupled with cluster mode for reliability and scalability, making them ideal for production workloads and automated pipelines.

Based on the video, **Partition Pruning** is an optimization technique that dramatically reduces the amount of data Spark has to read from the disk.

### **The Concept: Folders instead of One Big File**
Imagine your data is stored in folders based on a column, like **Department**:
*   `/Data/HR`
*   `/Data/IT`
*   `/Data/Finance`

### **How it Works**
*   **Without Partition Pruning:** If you run a query like `df.filter(Department == 'HR')`, Spark normally scans **all 10 GB** of data across all folders to find the "HR" records.
*   **With Partition Pruning:** Spark is smart enough to look at your filter (`Department == 'HR'`) and go **directly to the HR folder**. It completely skips the IT and Finance folders.

### **The Result**
Instead of scanning 10 GB, Spark might only scan **1 GB**. This makes your job significantly faster because it avoids reading unnecessary files,.

There is an even more advanced version of this called **Dynamic Partition Pruning (DPP)** that works with Joins. Would you like to see how that works?

Dynamic Partition Pruning (DPP) optimizes large-scale joins by "injecting" a filter from a small table into the scan of a large table *before* the join actually happens, preventing Spark from reading unnecessary data.

Here is how it works:

*   **The Scenario:** Imagine joining a small Dimension table (filtered for `Department = 'HR'`) with a massive Fact table (partitioned by Department). Without DPP, Spark would scan the **entire** massive Fact table (HR, IT, Finance, etc.) before joining.
*   **The Optimization:** With DPP, Spark "broadcasts" the filter (`HR`) from the small table to the large table's scan operation. Spark then **only reads the HR partition** of the large table and skips all others (IT, Finance) entirely.
*   **The Result:** This drastically reduces Disk I/O. As the guide notes, you might scan only **1 GB** of data instead of **10 GB**, significantly speeding up the query,.

**Constraint:** For DPP to activate, the column you filter on must be the same as the **partitioning column** of the large table.

Would you like to explore how **Adaptive Query Execution (AQE)** can further optimize these joins by dynamically coalescing partitions during runtime?

In [0]:
df1 = spark.read.table('testdb.testschema.healthcare_dataset')

display(df1)

In [0]:
# Step : Write DataFrame using partitioning to Unity Catalog table
# Rename columns with invalid characters (spaces) to valid names
valid_col_names = [
    ('Blood Type', 'Blood_Type'),
    ('Medical Condition', 'Medical_Condition'),
    ('Date of Admission', 'Date_of_Admission'),
    ('Insurance Provider', 'Insurance_Provider'),
    ('Billing Amount', 'Billing_Amount'),
    ('Room Number', 'Room_Number'),
    ('Admission Type', 'Admission_Type'),
    ('Discharge Date', 'Discharge_Date'),
    ('Test Results', 'Test_Results')
]

for old, new in valid_col_names:
    df1 = df1.withColumnRenamed(old, new)

# Write to Unity Catalog table

# Partition by Gender

df1.write \
  .mode("overwrite") \
  .partitionBy("Gender") \
  .saveAsTable("testdb.testschema")

In [0]:
# Step : Write DataFrame using partitioning to Unity Catalog table
# Rename columns with invalid characters (spaces) to valid names
valid_col_names = [
    ('Blood Type', 'Blood_Type'),
    ('Medical Condition', 'Medical_Condition'),
    ('Date of Admission', 'Date_of_Admission'),
    ('Insurance Provider', 'Insurance_Provider'),
    ('Billing Amount', 'Billing_Amount'),
    ('Room Number', 'Room_Number'),
    ('Admission Type', 'Admission_Type'),
    ('Discharge Date', 'Discharge_Date'),
    ('Test Results', 'Test_Results')
]

for old, new in valid_col_names:
    df1 = df1.withColumnRenamed(old, new)

# Write to Unity Catalog table

# Partition by Gender

df1.write \
  .mode("overwrite") \
  .saveAsTable("testdb.testschema.OutputwithoutData")


referred the ansh guid

# dyanmaic partiioning prusing

df_join_new = df_withoutpart.join(df_withpart, (df_withoutpart['name']==df_withpart['name']) & (df_withoutpart['department']==df_withpart['department']), how='inner')